Project Name : Group_DNA

Name :Mayuri Nilange

Batch :DS_DA_Oct2026

In [9]:
import numpy as np
from datetime import datetime, timedelta

file_path = "/content/hostel_bois.txt"

messages = []
system_messages = []
deleted_messages = {}
media_count = 0

with open(file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

current_message = None

for line in lines:
    line = line.rstrip("\n")

    if line == "":
        continue

    if " - " in line:
        parts = line.split(" - ", 1)

        try:
            timestamp = datetime.strptime(
                parts[0], "%d/%m/%y, %H:%M"
            )
        except ValueError:
            if current_message is not None:
                current_message["text"] += "\n" + line
            continue

        if current_message is not None:
            if current_message["type"] == "message":
                messages.append(current_message)
            else:
                system_messages.append(current_message)

        content = parts[1]

        if ": " in content:
            sender, message = content.split(": ", 1)

            current_message = {
                "timestamp": timestamp,
                "sender": sender,
                "text": message,
                "type": "message"
            }
        else:
            current_message = {
                "timestamp": timestamp,
                "sender": "System",
                "text": content,
                "type": "system"
            }

    else:
        if current_message is not None:
            current_message["text"] += "\n" + line

if current_message is not None:
    if current_message["type"] == "message":
        messages.append(current_message)
    else:
        system_messages.append(current_message)

for message in messages:
    if "This message was deleted" in message["text"]:
        sender = message["sender"]
        if sender not in deleted_messages:
            deleted_messages[sender] = 0
        deleted_messages[sender] += 1

    if "<Media omitted>" in message["text"]:
        media_count += 1

print("PARSER RESULTS")
print("----------------------")
print("Total lines:", len(lines))
print("Total messages:", len(messages))
print("System messages:", len(system_messages))
print("Deleted messages:", sum(deleted_messages.values()))
print("Media messages:", media_count)
print("\nFirst 3 messages:")

for message in messages[:3]:
    print(message)

PARSER RESULTS
----------------------
Total lines: 3178
Total messages: 3174
System messages: 4
Deleted messages: 15
Media messages: 32

First 3 messages:
{'timestamp': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'text': 'scene fix', 'type': 'message'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'text': 'haan', 'type': 'message'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 18), 'sender': 'Rahul', 'text': 'kya scene', 'type': 'message'}


In [10]:
# AI-assisted: Used ChatGPT to understand the group statistics logic.

participants = []
message_counts = {}
daily_counts = {}
hourly_counts = {}

for message in messages:
    sender = message["sender"]

    if sender not in participants:
        participants.append(sender)
        message_counts[sender] = 0

    message_counts[sender] += 1

    day = message["timestamp"].strftime("%d-%m-%Y")
    hour = message["timestamp"].hour

    if day not in daily_counts:
        daily_counts[day] = 0
    daily_counts[day] += 1

    if hour not in hourly_counts:
        hourly_counts[hour] = 0
    hourly_counts[hour] += 1

total_messages = len(messages)

print("========== GROUPDNA OVERVIEW ==========")
print("Total messages:", total_messages)
print("Total participants:", len(participants))
print("Total system messages:", len(system_messages))
print("Deleted messages:", sum(deleted_messages.values()))
print("Media messages:", media_count)

print("\n========== PARTICIPANT STATISTICS ==========")

for person in participants:
    percentage = (message_counts[person] / total_messages) * 100
    print(person, ":", message_counts[person],
          "messages |", round(percentage, 2), "%")

busiest_day = max(daily_counts, key=daily_counts.get)
busiest_hour = max(hourly_counts, key=hourly_counts.get)

print("\nBusiest day:", busiest_day,
      "-", daily_counts[busiest_day], "messages")

print("Busiest hour:", busiest_hour,
      ":00 -", busiest_hour + 1, ":00")

========== GROUPDNA OVERVIEW ==========
Total messages: 3174
Total participants: 6
Total system messages: 4
Deleted messages: 15
Media messages: 32

========== PARTICIPANT STATISTICS ==========
Rahul : 953 messages | 30.03 %
Priya : 718 messages | 22.62 %
Karan : 354 messages | 11.15 %
Neha : 635 messages | 20.01 %
Aman : 490 messages | 15.44 %
Vikas : 24 messages | 0.76 %

Busiest day: 04-05-2024 - 76 messages
Busiest hour: 18 :00 - 19 :00


In [11]:
# AI-assisted: Used ChatGPT to understand word-frequency analysis.

stop_words = (
    "the", "is", "are", "am", "a", "an", "and", "or",
    "to", "of", "in", "on", "for", "with", "this",
    "that", "it", "i", "you", "he", "she", "we",
    "they", "my", "your", "me", "was", "be", "at",
    "have", "has", "had", "do", "does", "did",
    "not", "but", "so", "if", "what", "when",
    "where", "who", "how", "from", "will"
)

word_counts = {}

for message in messages:
    text = message["text"]

    if "This message was deleted" in text:
        continue

    if "<Media omitted>" in text:
        continue

    words = text.lower().split()

    for word in words:
        word = word.strip(".,!?;:\"'()[]{}<>*_-/\\")

        if word != "" and word not in stop_words:
            if word not in word_counts:
                word_counts[word] = 0

            word_counts[word] += 1

sorted_words = sorted(
    word_counts.items(),
    key=lambda item: item[1],
    reverse=True
)

print("========== TOP 20 GROUP WORDS ==========")

for word, count in sorted_words[:20]:
    print(word, ":", count)


========== TOP 20 GROUP WORDS ==========
guys : 318
about : 274
hai : 268
today : 257
his : 217
just : 208
which : 202
everyone : 187
telling : 179
up : 172
bhai : 160
one : 157
started : 150
no : 146
scene : 145
entire : 145
please : 141
anyone : 139
yaar : 139
kya : 133


In [12]:
# AI-assisted: Used ChatGPT to understand NumPy activity matrices.

participants.sort()

activity_matrix = np.zeros((len(participants), 24), dtype=int)

for message in messages:
    sender = message["sender"]
    hour = message["timestamp"].hour

    if sender in participants:
        person_index = participants.index(sender)
        activity_matrix[person_index][hour] += 1

print("========== GROUP ACTIVITY HEATMAP ==========")
print("Rows = Participants | Columns = Hours (00-23)\n")

print("Name".ljust(12), end="")

for hour in range(24):
    print(str(hour).rjust(4), end="")

print()

for i in range(len(participants)):
    print(participants[i].ljust(12), end="")

    for hour in range(24):
        print(str(activity_matrix[i][hour]).rjust(4), end="")

    print()

print("\nTotal messages in matrix:", np.sum(activity_matrix))

========== GROUP ACTIVITY HEATMAP ==========
Rows = Participants | Columns = Hours (00-23)

Name           0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23
Aman          54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7  16   8  13  11   0  56
Karan          0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27  25  32  23  14   9   8
Neha           0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47  62  50  45  27  28  30
Priya          0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40  38  60  43  32  18   9
Rahul          3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49 105  76  41  92  60  54
Vikas          0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3   2   2   1   1   1   2

Total messages in matrix: 3174


In [13]:
# AI-assisted: Used ChatGPT to understand response-time and inactivity calculations.

messages.sort(key=lambda item: item["timestamp"])

response_times = {}

for person in participants:
    response_times[person] = []

# Calculate response time between consecutive messages from different people
for i in range(1, len(messages)):
    previous = messages[i - 1]
    current = messages[i]

    if previous["sender"] != current["sender"]:
        gap = current["timestamp"] - previous["timestamp"]
        minutes = gap.total_seconds() / 60

        # Consider gaps up to 24 hours as potential responses
        if minutes > 0 and minutes <= 1440:
            response_times[current["sender"]].append(minutes)

print("========== RESPONSE SPEED ==========")

for person in participants:
    times = response_times[person]

    if len(times) > 0:
        average = sum(times) / len(times)
        print(person, ":", round(average, 2),
              "minutes average response time")
    else:
        print(person, ": No response data")

# Calculate daily activity
daily_activity = {}

for message in messages:
    day = message["timestamp"].strftime("%d/%m/%y")

    if day not in daily_activity:
        daily_activity[day] = {}

    daily_activity[day][message["sender"]] = 1

# Generate all dates in the dataset
start_date = min(
    messages, key=lambda item: item["timestamp"]
)["timestamp"]

end_date = max(
    messages, key=lambda item: item["timestamp"]
)["timestamp"]

current_date = datetime.strptime(
    start_date.strftime("%d/%m/%y"), "%d/%m/%y"
)

last_date = datetime.strptime(
    end_date.strftime("%d/%m/%y"), "%d/%m/%y"
)

all_dates = []

while current_date <= last_date:
    all_dates.append(current_date.strftime("%d/%m/%y"))
    current_date += timedelta(days=1)

print("\n========== SILENT STREAK ANALYSIS ==========")

for person in participants:
    silent_days = 0
    current_streak = 0
    longest_streak = 0

    for day in all_dates:
        if day not in daily_activity or person not in daily_activity[day]:
            silent_days += 1
            current_streak += 1

            if current_streak > longest_streak:
                longest_streak = current_streak
        else:
            current_streak = 0

    percentage = (silent_days / len(all_dates)) * 100

    print(person)
    print("Silent days:", silent_days)
    print("Silent percentage:", round(percentage, 2), "%")
    print("Longest silent streak:", longest_streak, "days\n")

========== RESPONSE SPEED ==========
Aman : 56.94 minutes average response time
Karan : 37.54 minutes average response time
Neha : 40.94 minutes average response time
Priya : 42.98 minutes average response time
Rahul : 36.51 minutes average response time
Vikas : 46.3 minutes average response time

========== SILENT STREAK ANALYSIS ==========
Aman
Silent days: 0
Silent percentage: 0.0 %
Longest silent streak: 0 days

Karan
Silent days: 0
Silent percentage: 0.0 %
Longest silent streak: 0 days

Neha
Silent days: 0
Silent percentage: 0.0 %
Longest silent streak: 0 days

Priya
Silent days: 0
Silent percentage: 0.0 %
Longest silent streak: 0 days

Rahul
Silent days: 0
Silent percentage: 0.0 %
Longest silent streak: 0 days

Vikas
Silent days: 44
Silent percentage: 73.33 %
Longest silent streak: 11 days



In [14]:
# AI-assisted: Used ChatGPT to understand personality scoring and exclusive assignment.

caring_words = (
    "okay", "safe", "eat", "sleep", "take care", "are you",
    "please", "reminder", "drink water", "don't forget"
)

laugh_words = ("lol", "lmao", "haha", "rofl", "lmfao")

metrics = {}

for person in participants:
    person_messages = []
    for message in messages:
        if message["sender"] == person:
            person_messages.append(message)

    total = len(person_messages)
    night_count = 0
    word_total = 0
    uppercase_count = 0
    caring_count = 0
    comedy_count = 0
    question_count = 0
    burst_lengths = []
    current_burst = 0

    for message in person_messages:
        msg = message["text"]

        if "<Media omitted>" in msg or "This message was deleted" in msg:
            continue

        words = msg.split()
        word_total += len(words)

        if message["timestamp"].hour in (23, 0, 1, 2, 3, 4):
            night_count += 1

        letters = ""
        for character in msg:
            if character.isalpha():
                letters += character

        if len(letters) >= 3 and letters.isupper():
            uppercase_count += 1
        elif msg.count("!") >= 2:
            uppercase_count += 1

        lower_msg = msg.lower()

        for keyword in caring_words:
            if keyword in lower_msg:
                caring_count += 1
                break

        for word in words:
            clean_word = word.lower().strip(".,!?;:\"'()[]{}")
            if clean_word in laugh_words:
                comedy_count += 1
                break

        if msg.strip().endswith("?"):
            question_count += 1

    # Calculate consecutive message bursts
    for message in messages:
        if message["sender"] == person:
            current_burst += 1
        else:
            if current_burst > 0:
                burst_lengths.append(current_burst)
                current_burst = 0

    if current_burst > 0:
        burst_lengths.append(current_burst)

    average_burst = 0
    if len(burst_lengths) > 0:
        average_burst = sum(burst_lengths) / len(burst_lengths)

    average_words = word_total / total if total > 0 else 0
    night_percentage = night_count / total * 100 if total > 0 else 0
    uppercase_percentage = uppercase_count / total * 100 if total > 0 else 0
    comedy_percentage = comedy_count / total * 100 if total > 0 else 0
    question_percentage = question_count / total * 100 if total > 0 else 0

    metrics[person] = {
        "burst": average_burst,
        "night": night_percentage,
        "words": average_words,
        "uppercase": uppercase_percentage,
        "caring": caring_count,
        "comedy": comedy_percentage,
        "questions": question_percentage
    }

# Calculate silent percentages using the earlier daily analysis
for person in participants:
    silent_days = 0

    for day in all_dates:
        if day not in daily_activity or person not in daily_activity[day]:
            silent_days += 1

    metrics[person]["ghost"] = silent_days / len(all_dates) * 100

# Build eligible candidates for each archetype
archetypes = [
    "Spammer", "Group Mom", "Night Owl", "Storyteller",
    "Drama Queen", "Ghost", "Comedian", "Question Master"
]

candidates = {}

for archetype in archetypes:
    candidates[archetype] = []

    for person in participants:
        score = 0

        if archetype == "Spammer" and metrics[person]["burst"] > 3:
            score = metrics[person]["burst"]

        elif archetype == "Group Mom" and metrics[person]["caring"] > 0:
            score = metrics[person]["caring"]

        elif archetype == "Night Owl" and metrics[person]["night"] > 60:
            score = metrics[person]["night"]

        elif archetype == "Storyteller" and metrics[person]["words"] > 30:
            score = metrics[person]["words"]

        elif archetype == "Drama Queen" and metrics[person]["uppercase"] > 30:
            score = metrics[person]["uppercase"]

        elif archetype == "Ghost" and metrics[person]["ghost"] > 60:
            score = metrics[person]["ghost"]

        elif archetype == "Comedian" and metrics[person]["comedy"] > 0:
            score = metrics[person]["comedy"]

        elif archetype == "Question Master" and metrics[person]["questions"] > 25:
            score = metrics[person]["questions"]

        if score > 0:
            candidates[archetype].append((person, score))

# Exclusive assignment: strongest eligible candidate gets each archetype.
assigned_people = []
assignments = {}

for archetype in archetypes:
    candidates[archetype].sort(key=lambda item: item[1], reverse=True)

    for person, score in candidates[archetype]:
        if person not in assigned_people:
            assignments[person] = archetype
            assigned_people.append(person)
            break

print("========== GROUPDNA PERSONALITY REPORT ==========")

for person in participants:
    print("\nParticipant:", person)
    print("Average burst:", round(metrics[person]["burst"], 2))
    print("Night Owl activity:", round(metrics[person]["night"], 2), "%")
    print("Average words:", round(metrics[person]["words"], 2))
    print("Uppercase / drama:", round(metrics[person]["uppercase"], 2), "%")
    print("Caring messages:", metrics[person]["caring"])
    print("Silent days:", round(metrics[person]["ghost"], 2), "%")
    print("Comedy messages:", round(metrics[person]["comedy"], 2), "%")
    print("Questions:", round(metrics[person]["questions"], 2), "%")
    print("GROUPDNA ARCHETYPE:", assignments.get(person, "Unassigned"))

========== GROUPDNA PERSONALITY REPORT ==========

Participant: Aman
Average burst: 2.72
Night Owl activity: 79.39 %
Average words: 4.96
Uppercase / drama: 0.0 %
Caring messages: 87
Silent days: 0.0 %
Comedy messages: 0.0 %
Questions: 6.53 %
GROUPDNA ARCHETYPE: Night Owl

Participant: Karan
Average burst: 1.23
Night Owl activity: 1.98 %
Average words: 55.6
Uppercase / drama: 0.0 %
Caring messages: 29
Silent days: 0.0 %
Comedy messages: 0.0 %
Questions: 0.0 %
GROUPDNA ARCHETYPE: Storyteller

Participant: Neha
Average burst: 2.57
Night Owl activity: 4.72 %
Average words: 5.22
Uppercase / drama: 62.2 %
Caring messages: 23
Silent days: 0.0 %
Comedy messages: 0.0 %
Questions: 6.14 %
GROUPDNA ARCHETYPE: Drama Queen

Participant: Priya
Average burst: 1.66
Night Owl activity: 1.25 %
Average words: 4.96
Uppercase / drama: 0.0 %
Caring messages: 459
Silent days: 0.0 %
Comedy messages: 0.0 %
Questions: 29.25 %
GROUPDNA ARCHETYPE: Group Mom

Participant: Rahul
Average burst: 4.52
Night Owl activit

In [15]:
# AI-assisted: Used ChatGPT to understand final report formatting.

print("\n")
print("=" * 65)
print("                  GROUPDNA")
print("          YOUR WHATSAPP GROUP, DECODED")
print("=" * 65)

print("\n[ GROUP OVERVIEW ]")
print("-" * 65)
print("Total Messages       :", total_messages)
print("Total Participants   :", len(participants))
print("System Messages      :", len(system_messages))
print("Deleted Messages     :", sum(deleted_messages.values()))
print("Media Messages       :", media_count)
print("Active Date Range    :", len(all_dates), "days")
print("Busiest Day          :", busiest_day)
print("Busiest Hour         :", str(busiest_hour) + ":00")

print("\n[ MEMBER CONTRIBUTIONS ]")
print("-" * 65)

for person in participants:
    count = message_counts[person]
    percentage = count / total_messages * 100
    print(person.ljust(15), str(count).rjust(6),
          "messages  |", str(round(percentage, 2)).rjust(6), "%")

print("\n[ TOP 10 GROUP WORDS ]")
print("-" * 65)

for word, count in sorted_words[:10]:
    print(word.ljust(20), count)

print("\n[ GROUP PERSONALITIES ]")
print("-" * 65)

for person in participants:
    print(person.ljust(15), "->", assignments.get(person, "Unassigned"))

print("\n[ SILENT STREAKS ]")
print("-" * 65)

for person in participants:
    silent_days = 0
    current_streak = 0
    longest_streak = 0

    for day in all_dates:
        if day not in daily_activity or person not in daily_activity[day]:
            silent_days += 1
            current_streak += 1

            if current_streak > longest_streak:
                longest_streak = current_streak
        else:
            current_streak = 0

    print(person.ljust(15),
          "Longest silence:", longest_streak, "days")

print("\n" + "=" * 65)
print("             END OF GROUPDNA REPORT")
print("=" * 65)



                  GROUPDNA
          YOUR WHATSAPP GROUP, DECODED

[ GROUP OVERVIEW ]
-----------------------------------------------------------------
Total Messages       : 3174
Total Participants   : 6
System Messages      : 4
Deleted Messages     : 15
Media Messages       : 32
Active Date Range    : 60 days
Busiest Day          : 04-05-2024
Busiest Hour         : 18:00

[ MEMBER CONTRIBUTIONS ]
-----------------------------------------------------------------
Aman               490 messages  |  15.44 %
Karan              354 messages  |  11.15 %
Neha               635 messages  |  20.01 %
Priya              718 messages  |  22.62 %
Rahul              953 messages  |  30.03 %
Vikas               24 messages  |   0.76 %

[ TOP 10 GROUP WORDS ]
-----------------------------------------------------------------
guys                 318
about                274
hai                  268
today                257
his                  217
just                 208
which                202
e

In [17]:

# AI-assisted: ChatGPT helped with dashboard presentation.
# All analytics are calculated using Python and NumPy.

# Make sure the activity matrix matches the participant order
participants.sort()
activity_matrix = np.zeros((len(participants), 24), dtype=int)

for message in messages:
    person_index = participants.index(message["sender"])
    hour = message["timestamp"].hour
    activity_matrix[person_index][hour] += 1

# Create member cards
member_cards = ""

for person in participants:
    count = message_counts[person]
    percentage = count / total_messages * 100
    archetype = assignments.get(person, "Unassigned")

    member_cards += f"""
    <div class="member">
        <div class="avatar">{person[0].upper()}</div>
        <h3>{person}</h3>
        <span class="role">{archetype}</span>
        <h2>{count:,}</h2>
        <p>Messages sent</p>
        <div class="progress">
            <div style="width:{percentage}%;"></div>
        </div>
        <small>{percentage:.1f}% contribution</small>
    </div>
    """

# Create top words
word_html = ""

for word, count in sorted_words[:10]:
    word_html += f"""
    <span class="word">{word}
    <b>{count}</b></span>
    """

# Create NumPy heatmap
heatmap_html = ""

max_activity = int(np.max(activity_matrix))

for i in range(len(participants)):
    heatmap_html += f"<tr><th>{participants[i]}</th>"

    for hour in range(24):
        value = int(activity_matrix[i][hour])

        if max_activity > 0:
            intensity = value / max_activity
        else:
            intensity = 0

        red = int(255 - intensity * 65)
        green = int(240 - intensity * 100)
        blue = int(255 - intensity * 25)

        heatmap_html += f"""
        <td style="background:rgb({red},{green},{blue})"
            title="{participants[i]} at {hour}:00 — {value} messages">
            {value}
        </td>
        """

    heatmap_html += "</tr>"

# Calculate silent streaks
silent_html = ""

for person in participants:
    silent_days = 0
    current_streak = 0
    longest_streak = 0

    for day in all_dates:
        if day not in daily_activity or person not in daily_activity[day]:
            silent_days += 1
            current_streak += 1

            if current_streak > longest_streak:
                longest_streak = current_streak
        else:
            current_streak = 0

    silent_html += f"""
    <tr>
        <td>{person}</td>
        <td>{silent_days}</td>
        <td>{longest_streak} days</td>
    </tr>
    """

# Calculate average response times
response_html = ""

for person in participants:
    times = response_times[person]

    if len(times) > 0:
        average = sum(times) / len(times)
        response_value = str(round(average, 2)) + " min"
    else:
        response_value = "No data"

    response_html += f"""
    <tr>
        <td>{person}</td>
        <td>{response_value}</td>
    </tr>
    """

# Create complete HTML dashboard
dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">

<title>GroupDNA Dashboard</title>

<style>
* {{
    box-sizing: border-box;
    margin: 0;
    padding: 0;
}}

body {{
    font-family: Arial, sans-serif;
    background: #fff8f3;
    color: #302943;
    padding: 25px;
}}

.container {{
    max-width: 1200px;
    margin: auto;
}}

header {{
    background: linear-gradient(120deg,#ff83a5,#b49aff,#78cafa);
    color: white;
    padding: 40px;
    border-radius: 25px;
    margin-bottom: 28px;
}}

header h1 {{
    font-size: 43px;
    margin-bottom: 10px;
}}

header p {{
    margin-top: 8px;
}}

.section-title {{
    font-size: 25px;
    margin: 32px 0 18px;
}}

.stats {{
    display: grid;
    grid-template-columns: repeat(auto-fit,minmax(180px,1fr));
    gap: 17px;
}}

.stat {{
    padding: 25px;
    border-radius: 20px;
    box-shadow: 0 5px 15px #0000000c;
}}

.stat h2 {{
    font-size: 30px;
    margin-bottom: 8px;
}}

.pink {{background:#ffe0e9;color:#ad365f;}}
.blue {{background:#dff2ff;color:#226c9b;}}
.yellow {{background:#fff0c9;color:#986600;}}
.green {{background:#dff6e9;color:#28784c;}}
.purple {{background:#eee5ff;color:#7045a7;}}
.peach {{background:#ffe7d8;color:#a95428;}}

.members {{
    display:grid;
    grid-template-columns:repeat(auto-fit,minmax(210px,1fr));
    gap:18px;
}}

.member {{
    background:white;
    padding:23px;
    border-radius:20px;
    text-align:center;
    box-shadow:0 5px 18px #0000000c;
}}

.avatar {{
    background:linear-gradient(135deg,#ff9ab5,#a99aff);
    color:white;
    width:55px;
    height:55px;
    line-height:55px;
    border-radius:50%;
    margin:0 auto 12px;
    font-size:24px;
    font-weight:bold;
}}

.role {{
    display:inline-block;
    background:#f0eaff;
    color:#7856b5;
    padding:7px 13px;
    border-radius:20px;
    font-size:13px;
    margin:12px 0;
}}

.member h2 {{
    color:#ed638b;
    font-size:28px;
}}

.member p,.member small {{
    color:#888;
    margin:6px;
}}

.progress {{
    background:#eee;
    height:9px;
    border-radius:20px;
    margin:13px 0;
    overflow:hidden;
}}

.progress div {{
    background:linear-gradient(90deg,#ff7b9c,#a78bfa);
    height:100%;
}}

.panel {{
    background:white;
    padding:23px;
    border-radius:20px;
    box-shadow:0 5px 18px #0000000c;
    overflow-x:auto;
}}

.word {{
    display:inline-block;
    background:#e9f4ff;
    color:#3978a5;
    padding:12px 17px;
    border-radius:25px;
    margin:7px;
}}

table {{
    border-collapse:collapse;
    width:100%;
    font-size:13px;
    text-align:center;
}}

th,td {{
    padding:11px 8px;
    border:1px solid #f0eaf2;
}}

th {{
    background:#f3eaff;
    color:#694a9b;
}}

.heatmap td {{
    min-width:38px;
    font-weight:bold;
    color:#4a3a55;
}}

.insight {{
    background:linear-gradient(120deg,#fff0d5,#ffe2ed);
    padding:25px;
    border-radius:20px;
    line-height:2.2;
}}

footer {{
    text-align:center;
    margin-top:35px;
    padding:20px;
    color:#888;
}}

@media(max-width:600px) {{
    body {{padding:12px;}}
    header h1 {{font-size:32px;}}
}}
</style>
</head>

<body>
<div class="container">

<header>
    <h1>GroupDNA ✨</h1>
    <p>Your WhatsApp Group, Decoded!</p>
    <p>60 Days • Group Memories • Communication Insights</p>
</header>

<h2 class="section-title">📊 Group Overview</h2>

<div class="stats">
    <div class="stat pink">
        <h2>{total_messages:,}</h2>
        <p>Total Messages 💬</p>
    </div>
    <div class="stat blue">
        <h2>{len(participants)}</h2>
        <p>Group Members 👥</p>
    </div>
    <div class="stat yellow">
        <h2>{len(all_dates)}</h2>
        <p>Days Analyzed 📅</p>
    </div>
    <div class="stat green">
        <h2>{media_count}</h2>
        <p>Media Messages 📸</p>
    </div>
    <div class="stat purple">
        <h2>{sum(deleted_messages.values())}</h2>
        <p>Deleted Messages 🗑️</p>
    </div>
    <div class="stat peach">
        <h2>{len(system_messages)}</h2>
        <p>System Events ⚙️</p>
    </div>
</div>

<h2 class="section-title">🏆 Member Personalities</h2>

<div class="members">
    {member_cards}
</div>

<h2 class="section-title">🕒 Group Activity Heatmap</h2>

<div class="panel">
    <p style="margin-bottom:15px;">
        Messages sent by each member during each hour of the day.
        Darker pink cells indicate higher activity.
    </p>

    <table class="heatmap">
        <tr>
            <th>Member</th>
            {''.join(f'<th>{hour}</th>' for hour in range(24))}
        </tr>
        {heatmap_html}
    </table>
    <p style="margin-top:12px;color:#888;">
        Hours: 0–23 | Data generated using NumPy
    </p>
</div>

<h2 class="section-title">💬 Most Used Group Words</h2>

<div class="panel">
    {word_html}
</div>

<h2 class="section-title">⚡ Group Highlights</h2>

<div class="insight">
    <p>🔥 <b>Busiest Day:</b> {busiest_day}
    ({daily_counts[busiest_day]} messages)</p>
    <p>⏰ <b>Busiest Hour:</b> {busiest_hour}:00 –
    {busiest_hour + 1}:00</p>
    <p>👥 <b>Total Participants:</b> {len(participants)}</p>
    <p>📨 <b>Heatmap Total:</b> {int(np.sum(activity_matrix))}</p>
</div>

<h2 class="section-title">⏱️ Average Response Time</h2>

<div class="panel">
    <table>
        <tr><th>Member</th><th>Average Response Time</th></tr>
        {response_html}
    </table>
    <p style="margin-top:12px;color:#888;">
        Estimated from consecutive messages by different participants,
        not confirmed reply threads.
    </p>
</div>

<h2 class="section-title">👻 Silent Streak Analysis</h2>

<div class="panel">
    <table>
        <tr>
            <th>Member</th>
            <th>Silent Days</th>
            <th>Longest Silent Streak</th>
        </tr>
        {silent_html}
    </table>
</div>

<footer>
    <h2>GroupDNA ❤️</h2>
    <p>Built with Python, NumPy, HTML & CSS</p>
    <p>WhatsApp Group Analytics Project</p>
</footer>

</div>
</body>
</html>
"""

with open("GroupDNA_dashboard.html", "w", encoding="utf-8") as file:
    file.write(dashboard_html)

print("GROUPDNA DASHBOARD CREATED SUCCESSFULLY!")
print("File: GroupDNA_dashboard.html")
print("Heatmap shape:", activity_matrix.shape)
print("Heatmap total:", int(np.sum(activity_matrix)))
print("Parsed message total:", total_messages)


GROUPDNA DASHBOARD CREATED SUCCESSFULLY!
File: GroupDNA_dashboard.html
Heatmap shape: (6, 24)
Heatmap total: 3174
Parsed message total: 3174


In [18]:

# Compact GroupDNA dashboard - 1500 x 900
# AI-assisted: HTML/CSS presentation.

# Member cards
member_html = ""

for person in participants:
    count = message_counts[person]
    percent = count / total_messages * 100
    role = assignments.get(person, "Unassigned")

    member_html += f"""
    <div class="member">
      <b>{person}</b>
      <span>{role}</span>
      <strong>{count:,}</strong>
      <small>{percent:.1f}% contribution</small>
    </div>
    """

# Heatmap
heat_html = ""

for i in range(len(participants)):
    heat_html += f"<tr><th>{participants[i]}</th>"

    for hour in range(24):
        value = int(activity_matrix[i][hour])
        intensity = value / max_activity if max_activity else 0
        opacity = 0.12 + intensity * 0.88

        heat_html += f"""
        <td style="background:rgba(239,91,148,{opacity:.2f})"
            title="{participants[i]} | {hour}:00 | {value} messages">
            {value}
        </td>
        """

    heat_html += "</tr>"

# Top words
words_html = ""

for word, count in sorted_words[:8]:
    words_html += f'<span class="word">{word} <b>{count}</b></span>'

# Response times
response_html = ""

for person in participants:
    times = response_times[person]

    if len(times) > 0:
        average = round(sum(times) / len(times), 1)
        value = str(average) + " min"
    else:
        value = "N/A"

    response_html += f"<p>{person}: <b>{value}</b></p>"

# Silent streaks
silent_html = ""

for person in participants:
    silent_days = 0
    streak = 0
    longest = 0

    for day in all_dates:
        if day not in daily_activity or person not in daily_activity[day]:
            silent_days += 1
            streak += 1
            if streak > longest:
                longest = streak
        else:
            streak = 0

    silent_html += f"<p>{person}: <b>{longest} days</b></p>"

# Final dashboard
html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>GroupDNA Dashboard</title>
<style>
* {{box-sizing:border-box;margin:0}}
body {{
  background:#fff8f3;
  font-family:Arial,sans-serif;
  color:#393047;
  padding:15px;
}}
.dashboard {{
  width:1500px;
  height:900px;
  overflow:hidden;
  margin:auto;
  padding:20px;
  background:#fffaf7;
  border-radius:20px;
}}
header {{
  height:85px;
  padding:14px 22px;
  border-radius:17px;
  color:white;
  background:linear-gradient(110deg,#ff7da5,#b49aff,#76cafa);
  display:flex;
  justify-content:space-between;
  align-items:center;
}}
header h1 {{font-size:32px}}
header p {{font-size:13px;margin-top:5px}}
.kpis {{
  display:grid;
  grid-template-columns:repeat(4,1fr);
  gap:12px;
  margin:13px 0;
  height:90px;
}}
.kpi {{
  padding:13px 18px;
  border-radius:13px;
}}
.kpi h2 {{font-size:27px;margin-bottom:5px}}
.pink {{background:#ffe0e9;color:#a93460}}
.blue {{background:#dff2ff;color:#246b9a}}
.yellow {{background:#fff0c9;color:#956400}}
.green {{background:#dff6e9;color:#28784c}}
.layout {{
  display:grid;
  grid-template-columns:330px 770px 320px;
  gap:15px;
  height:635px;
}}
.panel {{
  background:white;
  border-radius:15px;
  padding:15px;
  box-shadow:0 3px 12px #0000000c;
  overflow:hidden;
}}
h3 {{font-size:17px;margin-bottom:12px}}
.members {{
  display:grid;
  grid-template-columns:1fr 1fr;
  gap:10px;
}}
.member {{
  background:#fff0f5;
  border-radius:12px;
  padding:13px 8px;
  text-align:center;
  height:126px;
}}
.member b {{display:block;font-size:15px}}
.member span {{
  display:block;
  color:#8a56a8;
  font-size:11px;
  margin:8px 0;
}}
.member strong {{
  display:block;
  font-size:22px;
  color:#e65c91;
}}
.member small {{font-size:10px;color:#777}}
table {{
  border-collapse:collapse;
  width:100%;
  table-layout:fixed;
  text-align:center;
  font-size:10px;
}}
td,th {{
  border:1px solid #f1e7ee;
  padding:9px 1px;
}}
th:first-child {{width:68px;background:#f2e8ff}}
td {{font-weight:bold;color:#65435b}}
.words {{margin-top:20px}}
.word {{
  display:inline-block;
  background:#e4f3ff;
  color:#3779a3;
  border-radius:15px;
  padding:9px 12px;
  margin:4px;
  font-size:12px;
}}
.highlight {{
  background:#fff0d8;
  border-radius:12px;
  padding:13px;
  line-height:2;
  font-size:13px;
  margin-bottom:15px;
}}
.small-panel {{
  border-top:1px solid #eee;
  padding-top:12px;
  margin-top:10px;
  font-size:12px;
  line-height:1.9;
}}
footer {{
  height:25px;
  text-align:center;
  padding-top:8px;
  color:#999;
  font-size:11px;
}}
</style>
</head>

<body>
<div class="dashboard">

<header>
 <div>
  <h1>GroupDNA ✨</h1>
  <p>Your WhatsApp Group, Decoded!</p>
 </div>
 <div>60 DAYS · 6 MEMBERS · GROUP WRAPPED</div>
</header>

<div class="kpis">
 <div class="kpi pink"><h2>{total_messages:,}</h2>Total Messages 💬</div>
 <div class="kpi blue"><h2>{len(participants)}</h2>Group Members 👥</div>
 <div class="kpi yellow"><h2>{len(all_dates)}</h2>Days Analyzed 📅</div>
 <div class="kpi green"><h2>{media_count}</h2>Media Messages 📸</div>
</div>

<div class="layout">

<section class="panel">
 <h3>🏆 Member Personalities</h3>
 <div class="members">{member_html}</div>
</section>

<section class="panel">
 <h3>🕒 24-Hour Activity Heatmap</h3>
 <table>
  <tr><th>Member</th>
   {''.join('<th>' + str(hour) + '</th>' for hour in range(24))}
  </tr>
  {heat_html}
 </table>

 <div class="words">
  <h3>💬 Most Used Words</h3>
  {words_html}
 </div>

 <div class="small-panel">
  <b>📨 Matrix total:</b> {int(np.sum(activity_matrix))}
  &nbsp; | &nbsp;
  <b>Deleted:</b> {sum(deleted_messages.values())}
  &nbsp; | &nbsp;
  <b>System:</b> {len(system_messages)}
 </div>
</section>

<section class="panel">
 <h3>⚡ Group Highlights</h3>
 <div class="highlight">
  🔥 <b>Busiest day:</b> {busiest_day}<br>
  ⏰ <b>Busiest hour:</b> {busiest_hour}:00<br>
  👥 <b>Members:</b> {len(participants)}
 </div>

 <div class="small-panel">
  <h3>⏱️ Response Time</h3>
  {response_html}
 </div>

 <div class="small-panel">
  <h3>👻 Longest Silent Streak</h3>
  {silent_html}
 </div>
</section>

</div>

<footer>
 GROUPDNA ❤️ &nbsp; | &nbsp; PYTHON + NUMPY + HTML + CSS
</footer>

</div>
</body>
</html>
"""

with open("GroupDNA_dashboard.html", "w", encoding="utf-8") as file:
    file.write(html)

print("1500 x 900 GroupDNA dashboard created!")
print("Heatmap:", activity_matrix.shape)
print("Matrix total:", int(np.sum(activity_matrix)))
print("Messages:", total_messages)


1500 x 900 GroupDNA dashboard created!
Heatmap: (6, 24)
Matrix total: 3174
Messages: 3174
